# Downstream RNA-seq analysis: Δpda1 vs. wild type


This notebook covers the **downstream** analysis of differential expression in a
*Saccharomyces cerevisiae* `pda1Δ` mutant relative to wild type, with a focus on
mitochondrial gene expression. Read alignment and differential expression testing
were performed upstream on the [Galaxy](https://usegalaxy.eu) platform; this
notebook begins from the exported DESeq2 results table and does not re-run
alignment or model fitting.

## Upstream processing (Galaxy)

**Alignment and quantification.** Clean paired-end reads were aligned with
RNA STAR (Dobin et al., 2013) to a custom *S. cerevisiae* reference genome in
which the S288c mitochondrial sequence (reference assembly R64,
NCBI GCF_000146045.2) was replaced by the assembled mitochondrial genome of the
LacO-tagged strain (yCO380). Per-gene read counts were generated using STAR's
`--quantMode GeneCounts` option with a matching gene-model annotation carrying
the manually curated LacO-mtDNA features.

**Sample exclusion.** One wild-type replicate (WT1) was excluded prior to
differential-expression analysis, as it segregated as a clear outlier from all
other samples by principal-component analysis. The remaining **two wild-type and
three Δpda1 replicates** were used throughout.

**Differential expression.** Tested with DESeq2 v1.40.2 (Love et al., 2014) under
a single-factor design (genotype), using the Wald test with Benjamini–Hochberg
correction and default parameters (median-of-ratios normalisation, parametric
dispersion fit). Analysis was run under R 4.3.1.


## What this notebook does

1. **Setup** — library imports and analysis environment
2. **Load and integrate** — joins the DESeq2 results with a summary table of mass
   spectrometry proteomics data, mitochondrial annotation, and mtDNA-associated
   measurements
3. **Classify genes** — labels each gene as mitochondrially encoded, a
   mitochondrial tRNA, a high-confidence mitochondrial protein, or nuclear encoded
4. **Export** — writes the processed dataframe and Supplementary Table S2
5. **Visualise** — volcano plots of differential expression, coloured by gene class

## Conventions used throughout

- **Fold change direction:** log2 fold changes are sign-inverted relative to the
  raw DESeq2 output, so that **positive values indicate higher transcript
  abundance in Δpda1 than in wild type**. This convention applies to every figure
  and exported table in this notebook.
- **Significance:** transcripts with a Benjamini–Hochberg adjusted p-value < 0.1
  are considered differentially expressed. Genes with `NA` adjusted p-values were
  removed by DESeq2 independent filtering and are retained at the end of the
  exported table.
- **Gene identifiers:** *S. cerevisiae* systematic names (e.g. `YER178W`), with
  standard gene symbols in the `GeneName` column. Note that mitochondrial
  features follow the custom LacO-mtDNA annotation described above.


## Running this notebook

Set `BASE_DIR` (input files) and `OUT_DIR` (supplementary table output) in the
loading cell to the directory containing the data files. Both default to the
notebook's own directory.

**Requires:** `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`, `openpyxl`


## 1. Setup

Analysis environment and library imports. This notebook performs downstream
analysis of differential expression results generated on the Galaxy Europe
server (STAR alignment → DESeq2), and does not re-run alignment or model
fitting itself.

| Library | Role in this notebook |
|---|---|
| `pandas` | Loading and reshaping the DESeq2 results table |
| `numpy` | Numerical operations and log transformations |
| `scipy.stats` | Statistical tests and distribution functions |
| `matplotlib`, `seaborn` | Figure generation 

In [ ]:
import os
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

## 2. Project paths

Input and output directories, defined relative to the repository root so the
notebook runs from a fresh clone without modification.

- `data/` — DESeq2 results tables and sample metadata exported from Galaxy
- `results/` — filtered gene lists and summary tables generated here
- `figures/` — publication figures written by this notebook

In [ ]:
from pathlib import Path

# Paths are relative to the notebook location so the analysis runs
# from a fresh clone of this repository.
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")

for d in (RESULTS_DIR, FIGURES_DIR):
    d.mkdir(exist_ok=True)

## 3. Load, integrate, and tidy datasets

Input files are read from `BASE_DIR`, which defaults to the notebook's own
directory. Set this to the location of the data files described in the
Data Availability statement.
Two inputs are combined into the working dataframe used for all downstream
analysis and figures:

1. **DESeq2 results** (`Results_withoutWT1.tabular`) — differential expression
   statistics exported from Galaxy, indexed by gene identifier.
2. **Integrated summary table** — mass spectrometry proteomics data, mitochondrial
   annotation, and mtDNA-associated measurements, joined per gene.

After concatenation, the columns required for downstream analysis are retained
and renamed:

| Column | Description |
|---|---|
| `Base_mean` | Mean normalised count across samples (DESeq2) |
| `log2FC` | Log2 fold change, sign-inverted — see below |
| `StdErr` | Standard error of the log2 fold change |
| `Wald_Stats` | Wald test statistic |
| `p_value` | Raw p-value |
| `p_adj` | Benjamini–Hochberg adjusted p-value |
| `GeneName` | Gene symbol |
| `Description` | Gene description |
| `Mito_Protein_Morgenstern` | Mitochondrial protein annotation (Morgenstern et al.) |
| `mtDNA_pulldown` | mtDNA pulldown enrichment measurement |
| `mtDNA_integrity` | mtDNA integrity measurement |
| `mtDNA_copynumber` | mtDNA copy number measurement |

**Fold change orientation.** The log2 fold change is multiplied by −1 so that
positive values represent depletion of PDA1 transcript. All figures and gene lists in
this notebook use the inverted convention. As the DESEQ2 was run comparing WT vs

**Sample exclusion.** The WT1 sample was excluded prior to model fitting. As it was not fitting the other WT samples in the PCA. 

In [ ]:
# Set to the directory containing the input files.
# '.' assumes the data files sit alongside this notebook.
BASE_DIR = '.'

genelist = pd.read_pickle(f'{BASE_DIR}/summary_with_dejanas_massspec_mw.pkl')
rnaResults = pd.read_csv(f'{BASE_DIR}/Results_withoutWT1.tabular', sep='\t', header=None, index_col=0)
combined = pd.concat([rnaResults, genelist], axis=1)

combined_2 = combined.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 14, 15, -15]]
df = combined_2

new_column_names = [
    'Base_mean',
    'log2FC',
    'StdErr',
    'Wald_Stats',
    'p_value',
    'p_adj',
    'GeneName',
    'Description',
    'Mito_Protein_Morgenstern',
    'mtDNA_pulldown',
    'mtDNA_integrity',
    'mtDNA_copynumber'
]

# Assign the new column names to the dataframe
df.columns = new_column_names

df['log2FC'] = df['log2FC'] * -1

## 4. Gene classification

Genes are annotated by their genomic origin and mitochondrial association,
using *S. cerevisiae* systematic gene names.

**Category definitions:**

| Flag | Criterion |
|---|---|
| `mtDNA_encoded` | Systematic name begins with `Q` (mitochondrial genome ORFs) |
| `mito_tRNA` | Systematic name begins with `t` (tRNA genes) |
| `Mito_Protein_Morgenstern` | Present in the Morgenstern et al. high-confidence mitochondrial proteome; missing values treated as absent |

**`Protein_Type` assignment.** Each gene receives a single label, assigned by
the following precedence:

1. `mito_tRNA` — if flagged as a tRNA gene
2. `Mito_Protein` — otherwise, if in the Morgenstern high-confidence set
3. `mtDNA_Encoded` — otherwise, if encoded on the mitochondrial genome
4. `Nuclear_encoded` — all remaining genes

Because the categories overlap, this precedence is part of the definition: a
mitochondrially encoded gene that also appears in the Morgenstern proteome is
labelled `Mito_Protein` rather than `mtDNA_Encoded`.

In [ ]:
df['mtDNA_encoded'] = df.index.str.startswith('Q')
df['mito_tRNA'] = df.index.str.startswith('t')
df['Mito_Protein_Morgenstern'] = df['Mito_Protein_Morgenstern'].fillna(False)
df['Mito_Protein_Morgenstern'] = df['Mito_Protein_Morgenstern'].astype(bool)
df['Protein_Type'] = df.apply(lambda row: 'mito_tRNA' if row['mito_tRNA'] 
                              else ('Mito_Protein' if row['Mito_Protein_Morgenstern'] 
                                    else ('mtDNA_Encoded' if row['mtDNA_encoded'] 
                                          else 'Nuclear_encoded')), axis=1)



## 5. Export processed data and supplementary tables

Writes the analysis-ready dataframe and the supplementary table accompanying
the manuscript.

**Outputs:**

- `Results_without_WT1_final_df.pkl` — the processed dataframe, serialised for
  reuse in downstream notebooks.
- `Supplementary_Table_S2_RNAseq.xlsx` — supplementary table containing:
  - **DESeq2 results (all)** — all genes with DESeq2 statistics, annotation
    columns, and classification, sorted by adjusted p-value in ascending order.
    Genes with `NA` adjusted p-values (removed by DESeq2 independent filtering)
    appear at the end of the table.
  - **mtDNA-encoded transcripts** — the subset of genes encoded on the
    mitochondrial genome, identified by systematic names beginning with `Q`.

The gene identifier index is reset to an explicit `systematic_name` column so
that identifiers are preserved in the exported file.

Fold changes in this table follow the inverted convention described in
Section 3: positive values indicate [TODO: direction].

In [ ]:
# Directory for manuscript supplementary tables.
OUT_DIR = '.'

df.to_pickle(f'{BASE_DIR}/Results_without_WT1_final_df.pkl')

# ---- export the RNA-seq DESeq2 results table (supplementary) ----
rnaseq_out = (df.reset_index()
              .rename(columns={"index": "systematic_name"})
              .sort_values("p_adj", ascending=True))

rnaseq_out.to_excel(f"{OUT_DIR}/Supplementary_Table_S2_RNAseq.xlsx",
                    index=False, sheet_name="DESeq2 results")
print(f"wrote Supplementary_Table_S2_RNAseq.xlsx  ({len(rnaseq_out)} genes)")

# mtDNA-encoded subset (systematic names starting with "Q")
mito_subset = rnaseq_out[rnaseq_out["systematic_name"].str.startswith("Q", na=False)]

with pd.ExcelWriter(f"{OUT_DIR}/Supplementary_Table_S2_RNAseq.xlsx", engine="openpyxl") as writer:
    rnaseq_out.to_excel(writer, index=False, sheet_name="DESeq2 results (all)")
    mito_subset.to_excel(writer, index=False, sheet_name="mtDNA-encoded transcripts")

print(f"wrote S2 with full table ({len(rnaseq_out)}) + mito subset ({len(mito_subset)})")

### Volcano plot — highlighted gene categories

Focused version of the volcano plot above, with selected gene categories
emphasised against a grey background of all remaining genes.

**Plotting scheme:**

- **Background layer:** all genes not matched by the highlight mapping, drawn
  in light grey.
- **Foreground layer:** genes matching the highlight categories, drawn on top
  in the project colour palette.
- **Gene labels:** restricted to a manually specified list of genes of interest
  (CHA1, ALT1, ILV5, ARG5,6, COX26, BAT1, YRO2, ARG8, VAR1), labelled only where
  they fall within the plotted axis range.
- **Dashed reference line:** −log10(0.05).
- **Axis limits:** x restricted to [−3, 3], y to [−1, 25]. Genes outside this
  range are not displayed; see the supplementary table for complete results.

Positive log2 fold change indicates higher transcript abundance in Δpda1
relative to wild type. Significance is assessed on Benjamini–Hochberg adjusted
p-values.

In [ ]:
Nupur2 = ["#88A27D","#b6c9c1","#BD5B28","#FFD30A","#4d6171","#162734"]
sns.color_palette(Nupur2)

df['Significant'] = df['p_adj'] < 0.1
df['-log10(p_value)'] = -np.log10(df['p_adj'])

nupur2_colors = sns.color_palette(Nupur2, 5)  # Assuming Nupur2 has at least 5 colors
custom_palette = [nupur2_colors[2], nupur2_colors[0], nupur2_colors[4], nupur2_colors[3]]

# Create the plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='log2FC',
    y='-log10(p_value)',
    s=100,  # Point size
    edgecolor='black', hue='Protein_Type', palette=custom_palette)

# Significance threshold, matching the p_adj < 0.1 cutoff used for labelling
plt.axhline(y=-np.log10(0.1), color='red', linestyle='--',
            label='Significance threshold (adjusted p = 0.1)')

x_min, x_max = -5, 5
y_min, y_max = 0, 25
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)

# Label significant points only if within axis limits
for _, row in df[df['Significant']].iterrows():
    if x_min <= row['log2FC'] <= x_max and y_min <= row['-log10(p_value)'] <= y_max:
        plt.text(
            row['log2FC'], row['-log10(p_value)'], row['GeneName'],
            fontsize=8, ha='right', va='bottom'
        )

# Add labels and title
plt.title('Volcano Plot of Transcript Fold Change vs Significance')
plt.xlabel('log2 fold change (Δpda1 / WT)')
plt.ylabel('-log10(adjusted p-value)')

plt.rcParams['svg.fonttype'] = 'none'
# plt.savefig('VolcanoPlot_allproteins.svg')
# plt.savefig('VolcanoPlot_allproteins.png')

plt.show()